# LOKLOK ? Tests for previously untested claims

Three model-internal tests:

1. **Filter ablation** ? leave-one-filter-out for Laplace, Lyapunov, DeGroot, Lagrange, Hilbert, Percolation, Entropy. (Hamiltonian is a diagnostic and does not enter the outcome.)
2. **Source/feature ablation** ? full vs quant-only vs no-corpora vs no-FGD vs no-self-report vs no-PO vs shuffled-text.
3. **Anchor bootstrap** ? 50 resamples of the 52 measured anchors, refitting DML/IPW, manifold, Q0, and rerunning the main simulator.

Run the cells **one at a time**; they are heavy. The final zip is unique: `loklok_v8_untested_claims_results.zip`.


In [ ]:
!nvidia-smi
import torch
print("torch", torch.__version__)
print("cuda", torch.cuda.is_available())
if torch.cuda.is_available():
    print(torch.cuda.get_device_name(0))


In [ ]:
!wget -q -O /content/colab_bundle_v8.zip https://cycles-bless-equal-printers.trycloudflare.com/colab_bundle_v8.zip
!cd /content && rm -rf loklok_v8 && unzip -q -o colab_bundle_v8.zip -d /content
from pathlib import Path
ROOT=Path('/content/loklok_v8')
print(ROOT, ROOT.exists())
for f in ['filter_ablation.py','source_ablation.py','anchor_bootstrap.py']:
    print(f, (ROOT/'02_code/pipeline'/f).exists())
print('cache:', (ROOT/'03_data/model_outputs/embeddings/embeddings_0f6c66e54fb6ca2c.npy').exists())


In [ ]:
# Test 1: leave-one-filter-out
!cd /content/loklok_v8 && PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True python 02_code/pipeline/filter_ablation.py


In [ ]:
# Test 2: source/feature ablation
!cd /content/loklok_v8 && PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True python 02_code/pipeline/source_ablation.py


In [ ]:
# Test 3: bootstrap over the 52 anchors (50 replicates)
!cd /content/loklok_v8 && PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True python 02_code/pipeline/anchor_bootstrap.py


In [ ]:
import json
from pathlib import Path
ROOT=Path('/content/loklok_v8')
out=ROOT/'03_data/model_outputs'
for name in ['filter_ablation.json','source_ablation.json','anchor_bootstrap.json']:
    p=out/name
    if p.exists():
        r=json.loads(p.read_text(encoding='utf-8'))
        print('\n===',name,'===')
        if 'paired_difference_vs_baseline' in r:
            for k,v in r['paired_difference_vs_baseline'].items(): print(k, round(v['mean'],3), [round(v['ci_low'],3),round(v['ci_high'],3)])
        if 'paired_difference_vs_full' in r:
            for k,v in r['paired_difference_vs_full'].items(): print(k, round(v['mean'],3), [round(v['ci_low'],3),round(v['ci_high'],3)])
        if 'bootstrap' in r:
            print(r['bootstrap'])


In [ ]:
!cd /content && rm -f loklok_v8_untested_claims_results.zip && zip -q -r loklok_v8_untested_claims_results.zip loklok_v8/03_data/model_outputs/filter_ablation.json loklok_v8/03_data/model_outputs/filter_ablation.csv loklok_v8/03_data/model_outputs/source_ablation.json loklok_v8/03_data/model_outputs/source_ablation.csv loklok_v8/03_data/model_outputs/anchor_bootstrap.json loklok_v8/03_data/model_outputs/anchor_bootstrap.csv
!ls -lh /content/loklok_v8_untested_claims_results.zip


In [ ]:
from google.colab import files
files.download('/content/loklok_v8_untested_claims_results.zip')
